# Prepare task 3 recovery images on CPU

This notebook prepares image lists for a future controlled recovery experiment. It does not train, unlearn, load checkpoints, or reserve GPU memory. In Colab, select **Runtime > Change runtime type > Hardware accelerator: None** before running cells in order.

For each of task 3's ten classes, initial training gets 400 images and a recovery reserve gets 100. Recovery budgets use the first 1, 5, and 20 reserved images per class. The 500 validation images stay separate. Existing checkpoints trained on all images cannot satisfy this split retrospectively.

The split is fixed across future training seeds 0, 1, and 2. The image-selection seed below is separate from both the class-partition seed (42) and model-training seeds. A JSON manifest records every selection. Only task 3 is prepared here; task Y and the full study protocol remain to be decided.

## Cell 1: Find the repository

On your computer, run from this repository. In a fresh Colab CPU session, this cell clones the public repository to local runtime storage if it is missing. It installs no packages. The recorded Git revision identifies the checkout used to prepare the lists.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib
import random
import shutil
import urllib.request
import zipfile
import tempfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/CARK")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").is_file()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from the CARK repository.")
CODE_REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Revision:", CODE_REVISION)
print("This notebook uses CPU and disk only.")


## Cell 2: Choose paths and the fixed image seed

Change `DATA_ROOT` if Tiny ImageNet is already extracted elsewhere. Local disk is preferable to Drive for indexing thousands of files. Download is off by default. Set `DOWNLOAD_DATA = True` only if the dataset is missing.

`OUTPUT_DIR` stores small lists, not copied images. In Colab, local files disappear when the runtime ends. Cell 7 now saves the manifest to R2 before cell 8 exports a ZIP. You can instead set `OUTPUT_DIR` to an already mounted Drive directory. R2 saving uses the existing Colab Secrets from notebook 15. No checkpoint downloads are needed. Set PUBLISH_R2=False only if you will preserve the ZIP yourself.

Keep `SPLIT_SEED` unchanged across the three future model-training seeds. Changing it creates a different data selection.

In [ ]:
TASK_ID = 3
SPLIT_SEED = 20261008
RESERVE_PER_CLASS = 100
BUDGETS = (1, 5, 20)
DOWNLOAD_DATA = False
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
OUTPUT_DIR = REPO / "outputs/recovery_setup/task3_split_v1"
PARTITION_PATH = REPO / "uncle/task_partition.json"
print("Dataset:", DATA_ROOT)
print("Output:", OUTPUT_DIR)

PUBLISH_R2 = True
BUCKET = "uncle-experiments"
R2_PREFIX = "uncle/recovery_study/recovery_task3_reserved_v1"


## Cell 3: Locate or download Tiny ImageNet

An existing dataset is reused. If download is enabled and the dataset is missing, this cell downloads the Stanford ZIP once and extracts it on CPU. This is the slowest setup step: it needs network access and disk space but gains nothing from a GPU. The archive includes the full dataset even though this notebook indexes only ten classes.

Extraction refuses to merge into an incomplete dataset directory. It does not delete existing data. If an earlier download was interrupted, choose a fresh parent directory in cell 2. The ZIP is retained for reuse.

In [ ]:
markers = ("wnids.txt", "train", "val/val_annotations.txt")
missing = [name for name in markers if not (DATA_ROOT / name).exists()]
if missing:
    if not DOWNLOAD_DATA:
        raise FileNotFoundError(f"Missing {missing} under {DATA_ROOT}. Set the correct DATA_ROOT or enable DOWNLOAD_DATA in cell 2.")
    if DATA_ROOT.name != "tiny-imagenet-200":
        raise ValueError("DATA_ROOT must end in tiny-imagenet-200 for downloading.")
    if DATA_ROOT.exists():
        raise FileExistsError("Dataset directory is incomplete. Choose a fresh DATA_ROOT; existing files will not be overwritten.")
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    archive = DATA_ROOT.parent / "tiny-imagenet-200.zip"
    if not archive.exists():
        if shutil.disk_usage(DATA_ROOT.parent).free < 3 * 1024**3:
            raise OSError("Allow at least 3 GiB of free disk space for download and extraction.")
        partial = archive.with_suffix(".zip.partial")
        urllib.request.urlretrieve("https://cs231n.stanford.edu/tiny-imagenet-200.zip", partial)
        partial.replace(archive)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            relative = Path(member.filename)
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "tiny-imagenet-200":
                raise ValueError("Unexpected archive path.")
        z.extractall(DATA_ROOT.parent)
    if any(not (DATA_ROOT / name).exists() for name in markers):
        raise FileNotFoundError("Extraction did not produce the expected dataset layout.")
print("Dataset is available. No images were loaded into memory.")


## Cell 4: Build deterministic image lists

Read task 3's classes from the repository's seed-42 partition. Their order defines labels 0 through 9. For each class, sort the filenames, then shuffle with an independent seed derived from the fixed split seed and class ID. Take 100 for the reserve and 400 for initial training.

The saved paths are relative to `DATA_ROOT`, so they also work after downloading the same dataset into another runtime. This cell reads filenames and validation labels, not image pixels. It checks the existence of the selected task's 5,500 images; it does not scan every task or hash image contents.

In [ ]:
partition_bytes = PARTITION_PATH.read_bytes()
partition = json.loads(partition_bytes)
if partition["seed"] != 42 or partition["classes_per_task"] != 10:
    raise ValueError("Expected the existing seed-42 ten-class partition.")
classes = next(group["wnids"] for group in partition["tasks"] if group["task_id"] == TASK_ID)
if len(classes) != 10 or len(set(classes)) != 10:
    raise ValueError("Expected ten unique classes.")
if not set(classes) <= set((DATA_ROOT / "wnids.txt").read_text().split()):
    raise ValueError("Task classes are missing from this dataset.")
validation = {wnid: [] for wnid in classes}
for line in (DATA_ROOT / "val/val_annotations.txt").read_text().splitlines():
    filename, wnid, *_ = line.split()
    if wnid in validation:
        candidates = [DATA_ROOT / "val/images" / filename,
                      DATA_ROOT / "val" / wnid / filename,
                      DATA_ROOT / "val" / wnid / "images" / filename]
        path = next((p for p in candidates if p.is_file()), None)
        if path is None:
            raise FileNotFoundError(filename)
        validation[wnid].append(path.relative_to(DATA_ROOT).as_posix())
rows = []
for label, wnid in enumerate(classes):
    folder = DATA_ROOT / "train" / wnid
    paths = sorted(folder.rglob("*.JPEG"), key=lambda p: p.relative_to(DATA_ROOT).as_posix())
    if len(paths) != 500 or len({p.name for p in paths}) != 500:
        raise ValueError(f"{wnid}: expected 500 unique training filenames, got {len(paths)} files.")
    seed = int.from_bytes(hashlib.sha256(f"{SPLIT_SEED}:{wnid}".encode()).digest(), "big")
    random.Random(seed).shuffle(paths)
    ordered = [p.relative_to(DATA_ROOT).as_posix() for p in paths]
    reserve = ordered[:RESERVE_PER_CLASS]
    rows.append({"wnid": wnid, "label": label,
                 "initial_train": ordered[RESERVE_PER_CLASS:],
                 "recovery_reserve": reserve,
                 "recovery": {str(b): reserve[:b] for b in BUDGETS},
                 "validation": sorted(validation[wnid])})
manifest = {"version": 1, "task_id": TASK_ID, "split_seed": SPLIT_SEED,
            "selection": "sorted relative paths; Python random.Random with SHA-256-derived per-class seed",
            "partition_sha256": hashlib.sha256(partition_bytes).hexdigest(),
            "reserve_per_class": RESERVE_PER_CLASS, "budgets": list(BUDGETS), "classes": rows}
print("Built image lists for", len(rows), "classes.")


## Cell 5: Check counts and separation

These checks stop the notebook if an image appears in both initial training and recovery, if validation overlaps training, or if a budget has the wrong size. A pass verifies filename selection and file existence. It does not prove that image contents have no duplicates or that a future training run will use the lists.

The printed table shows counts for every class. No model or GPU is involved.

In [ ]:
all_train, all_val = [], []
for row in rows:
    train, reserve, val = map(set, (row["initial_train"], row["recovery_reserve"], row["validation"]))
    assert len(train) == 400 and len(reserve) == 100 and len(val) == 50
    assert train.isdisjoint(reserve) and (train | reserve).isdisjoint(val)
    for b in BUDGETS:
        assert row["recovery"][str(b)] == row["recovery_reserve"][:b]
        assert len(set(row["recovery"][str(b)])) == b
    for name in train | reserve | val:
        assert (DATA_ROOT / name).is_file(), name
    all_train.extend(train | reserve)
    all_val.extend(val)
assert len(all_train) == len(set(all_train)) == 5000
assert len(all_val) == len(set(all_val)) == 500
print("Class       Label   Initial   Reserve   Validation   Recovery budgets")
for row in rows:
    print(f'{row["wnid"]:12} {row["label"]:5} {len(row["initial_train"]):9} {len(row["recovery_reserve"]):9} {len(row["validation"]):12}   1 / 5 / 20')
print("PASS: 4,000 initial-training images; 1,000 reserved; 500 validation.")
print("Recovery totals: 10, 50, and 200 images.")


## Cell 6: Save and reload the manifest

Save the JSON lists, their SHA-256 fingerprint (a file-content identifier), and preparation metadata. Rerunning with the same split reuses the existing manifest. A different split is refused rather than overwriting it; choose a new output directory for a deliberately different split.

The final readback checks that the saved lists match memory. A future training report should record this fingerprint and use these exact lists.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
payload = (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode("utf-8")
manifest_path = OUTPUT_DIR / "manifest.json"
if manifest_path.exists():
    if manifest_path.read_bytes() != payload:
        raise FileExistsError("An incompatible manifest exists. Choose a new OUTPUT_DIR.")
else:
    with manifest_path.open("xb") as stream:
        stream.write(payload)
fingerprint = hashlib.sha256(payload).hexdigest()
metadata = {"code_revision": CODE_REVISION, "python": sys.version,
            "manifest_sha256": fingerprint, "dataset_root": str(DATA_ROOT),
            "checks": "counts, disjoint paths, nested budgets, and file existence passed",
            "scope": "CPU image-list preparation only; no study models trained"}
metadata_path = OUTPUT_DIR / "preparation.json"
if not metadata_path.exists():
    with metadata_path.open("x", encoding="utf-8") as stream:
        json.dump(metadata, stream, indent=2)
        stream.write("\n")
assert json.loads(manifest_path.read_text()) == manifest
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == fingerprint
print("Saved:", manifest_path)
print("SHA-256:", fingerprint)
print("Metadata:", metadata_path)


## Cell 7: Save the lists to R2 before disconnecting

With PUBLISH_R2=True, save manifest.json and preparation.json to your existing private bucket. In Colab, enable notebook access to R2_ENDPOINT, R2_ACCESS_KEY_ID, and R2_SECRET_ACCESS_KEY in the Secrets panel, as in notebook 15. Credentials are not printed or saved. Only the small JSON files are uploaded; no dataset images or checkpoints are transferred.

The upload reads each object back to verify its contents. Existing identical files are reused. Different files at the same key are refused rather than overwritten. Wait for the final confirmation before disconnecting. Notebook 17 can then download the manifest directly from R2. This cell does not create notebook 17's protocol.json; its CPU PREPARE mode does that.


In [ ]:
if PUBLISH_R2:
    import os
    import importlib.util
    if importlib.util.find_spec("boto3") is None:
        if not IN_COLAB:
            raise ImportError("Install boto3 for R2 saving, or preserve the ZIP with PUBLISH_R2=False.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3>=1.36"], check=True)
    sys.path.insert(0, str(REPO))
    from uncle.storage import R2Store
    def storage_secret(name):
        value = os.environ.get(name)
        if not value and IN_COLAB:
            from google.colab import userdata
            value = userdata.get(name)
        if not value:
            raise RuntimeError("Missing storage secret: " + name + ". The local manifest is saved, but R2 saving has not completed.")
        return value
    store = R2Store.connect(storage_secret("R2_ENDPOINT"), BUCKET,
                            storage_secret("R2_ACCESS_KEY_ID"), storage_secret("R2_SECRET_ACCESS_KEY"))
    for path in (manifest_path, metadata_path):
        key = R2_PREFIX + "/" + path.name
        store.upload(path, key)
        print("Verified in R2:", key)
    print("Manifest preserved in R2. You can disconnect after exporting the optional ZIP.")
else:
    print("R2 saving is off. Preserve the ZIP from the next cell before disconnecting.")


## Cell 8: Preserve the lists outside Colab

This makes a small ZIP containing only the manifest and metadata. It does not copy the dataset or models. In Colab, download the ZIP before ending the CPU session. On your computer, keep it with the study records. Loading the manifest in a later session does not require repeating the shuffle.

In [ ]:
bundle_path = OUTPUT_DIR / "recovery_setup.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(manifest_path, "manifest.json")
    bundle.write(metadata_path, "preparation.json")
print("Archive:", bundle_path, "bytes:", bundle_path.stat().st_size)
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle_path))


## Cell 9: Define a loader for later training (optional)

Run this cell only when PyTorch, torchvision, and Pillow are already installed. Earlier cells need only Python's standard library. This cell defines a function; it does not read image pixels, construct a model, move tensors to a GPU, or start training.

The loader reads the saved paths and supplies labels 0 through 9. Pixels use RGB and values in [0, 1], matching the current preprocessing. Zero worker processes avoid extra memory and notebook process overhead. Images are opened one at a time when a future training loop requests a batch.

For task 3, replace the ordinary `build_tasks()` training dataset with `make_dataset("initial_train")`. Keep `make_dataset("validation")` for evaluation. The never-learned control skips task 3 completely during initial learning. Other tasks need their separately agreed study setup. Never pass `recovery_reserve` into initial training.

In [ ]:
def make_dataset(group, budget=None, root=DATA_ROOT, saved_manifest=manifest_path):
    from PIL import Image
    from torch.utils.data import Dataset
    from torchvision.transforms import ToTensor
    saved = json.loads(Path(saved_manifest).read_text())
    if group not in ("initial_train", "validation", "recovery"):
        raise ValueError("Choose initial_train, validation, or recovery.")
    if group == "recovery" and budget not in saved["budgets"]:
        raise ValueError("Recovery budget must be one of the saved budgets.")
    samples = []
    for row in saved["classes"]:
        paths = row["recovery"][str(budget)] if group == "recovery" else row[group]
        samples.extend((Path(root) / name, row["label"]) for name in paths)
    class ManifestDataset(Dataset):
        def __init__(self):
            self.targets = [label for _, label in samples]
            self.classes = [row["wnid"] for row in saved["classes"]]
            self.task_id = saved["task_id"]
            self.transform = ToTensor()
        def __len__(self):
            return len(samples)
        def __getitem__(self, index):
            path, label = samples[index]
            with Image.open(path) as source:
                image = source.convert("RGB")
            return self.transform(image), label
    return ManifestDataset()

# Later training code can use:
# task3 = {"train": make_dataset("initial_train"),
#          "test": make_dataset("validation")}
# recovery_data = make_dataset("recovery", budget=5)
# DataLoader(recovery_data, batch_size=10, num_workers=0, ...)
print("Loader defined. No images or models loaded.")


## Finished: what is ready and what remains

Ready: fixed task-3 image selections, checked counts and separation, a portable manifest, and an optional dataset loader.

Before GPU training, agree on task Y, the readiness criterion, the matched request sequences, and the recovery settings. The future runner must use the saved initial-training list, record the manifest fingerprint, and build fresh study models. This notebook does not make existing checkpoints eligible for the reserved-data study. Stop the CPU session after confirming the R2 upload or preserving the ZIP.